# Painting Diffusion — a class-conditional DDPM trained from scratch on a MacBook

A 23M parameter denoising diffusion model trained end-to-end on an M2 Pro in
about **3.7 hours**, on **5,202 paintings** by Van Gogh, Renoir, Monet and
Cézanne pulled from WikiArt. No pretrained weights, no latent space, no cloud —
the UNet, the noise schedule, the training loop and the sampler are all here.

It is conditioned on **two independent axes**, so you can ask for a specific
*kind* of image rather than just sampling the whole distribution:

| axis | values |
|---|---|
| **artist** | `van_gogh`, `renoir`, `monet`, `cezanne` |
| **genre**  | `landscape`, `portrait`, `still_life`, `cityscape`, `genre_painting`, `other` |

```python
sample(artist="van_gogh", genre="landscape", w_artist=3.0, w_genre=2.0)
```

Each axis gets its own classifier-free-guidance strength, and you can blend
artists (`0.5 monet + 0.5 van_gogh`) at sampling time.

**Run order:** run every cell top to bottom. Section 2 downloads ~7.3 GB once
and caches to disk; every later run skips it. Section 8 (training) is
checkpointed, so you can interrupt the kernel and re-run the cell to resume.

## 1. Setup

In [ ]:
import os, io, json, math, time, random, shutil, collections
from pathlib import Path
from dataclasses import dataclass

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm

torch.backends.mps.is_available() or print("!! MPS unavailable - will fall back to CPU (very slow)")
DEVICE = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

def seed_everything(s=0):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)

def sync():
    if DEVICE.type == "mps":
        torch.mps.synchronize()

seed_everything(0)
print("torch", torch.__version__, "| device:", DEVICE)

## 2. Configuration

Everything tunable lives here.

Why these four painters: WikiArt ships as 72 parquet shards totalling ~34 GB, but
these four artists happen to be concentrated in just **16** of those shards.
Fetching only those shards costs ~7.3 GB instead of 34 GB. Pissarro and Gauguin
were dropped precisely because they are smeared across 58 shards — 1,200 extra
paintings would have cost another 19 GB.

In [ ]:
# ---- dataset -----------------------------------------------------------
ARTISTS = {22: "van_gogh", 17: "renoir", 4: "monet", 16: "cezanne"}   # WikiArt label id -> name
ARTIST_NAMES = [ARTISTS[k] for k in sorted(ARTISTS)]
ARTIST_ID = {n: i for i, n in enumerate(ARTIST_NAMES)}

GENRES = ["landscape", "portrait", "still_life", "cityscape", "genre_painting", "other"]
GENRE_ID = {n: i for i, n in enumerate(GENRES)}
MIN_GENRE_COUNT = 120      # rarer genres get folded into "other"

N_ARTISTS, N_GENRES = len(ARTIST_NAMES), len(GENRES)
NULL_ARTIST, NULL_GENRE = N_ARTISTS, N_GENRES    # extra embedding slots used for CFG

# ---- resolution --------------------------------------------------------
IMAGE_SIZE   = 64     # what the model trains on
CACHE_SIZE   = 80     # cached square size; randomly cropped down to IMAGE_SIZE
ARCHIVE_SIDE = 144    # JPEG archive short side, big enough to rebuild a 128px cache later

# ---- paths -------------------------------------------------------------
ROOT     = Path(".").resolve()
DATA     = ROOT / "data";        DATA.mkdir(exist_ok=True)
ARCHIVE  = DATA / "paintings";   ARCHIVE.mkdir(exist_ok=True)
TMP      = DATA / "_tmp_shards"
CKPT_DIR = ROOT / "checkpoints"; CKPT_DIR.mkdir(exist_ok=True)
OUT_DIR  = ROOT / "samples";     OUT_DIR.mkdir(exist_ok=True)
CACHE_IMG, CACHE_LAB = DATA / "cache_images.npy", DATA / "cache_labels.npy"
MANIFEST = DATA / "manifest.json"

# ---- model -------------------------------------------------------------
# Sized empirically for 16GB of unified memory. The width at the FULL 64x64
# level is what drives activation memory, so we stay narrow there (64) and put
# the capacity in the deeper, cheaper levels via the 1-2-3-4 multipliers.
# base 128 + (1,2,2,4) would be 77M params and ~20GB -> instant OOM. See the
# memory table in section 6 before changing these.
BASE_CH      = 64
CHANNEL_MULT = (1, 2, 3, 4)     # 64ch@64px -> 128@32 -> 192@16 -> 256@8
NUM_RES      = 2
ATTN_RES     = (16, 8)
NUM_HEADS    = 4
DROPOUT      = 0.1

# ---- diffusion ---------------------------------------------------------
TIMESTEPS = 1000
PRED_TYPE = "v"                 # "v" or "eps"; v is noticeably more stable at this scale

# ---- training ----------------------------------------------------------
# Batch 16 is a memory choice, not a speed choice: throughput is nearly flat from
# 16 to 32, but batch 64 crosses a cliff where MPS starts swapping and a step goes
# from 0.5s to 54s. 16 leaves room for a browser to be open. Measured, section 7.
BATCH_SIZE  = 16
LR          = 2e-4
WARMUP      = 1000
TOTAL_STEPS = 24_000        # ~3.7 hours at ~0.55 s/step
GRAD_CLIP   = 1.0
EMA_DECAY   = 0.9995
SAMPLE_EVERY, CKPT_EVERY, LOG_EVERY = 1000, 1000, 50

print(f"{N_ARTISTS} artists x {N_GENRES} genres, {IMAGE_SIZE}x{IMAGE_SIZE}px")

## 3. Build the dataset

Three steps, all skipped automatically once `data/cache_images.npy` exists:

1. **Scan** the `artist` column of all 72 shards (~3 s — it is a tiny int column,
   so this is cheap even though the files are 470 MB each) to find which shards
   hold our painters.
2. **Download** only those shards, extract matching paintings, save each as a
   JPEG with short side 144, then **delete the shard** so peak disk stays ~500 MB.
3. **Pack** the JPEGs into one `uint8` array held in RAM during training.

Step 2 is resumable per shard: interrupt it and re-run, and it picks up where it
stopped.

In [ ]:
import pyarrow.parquet as pq
from huggingface_hub import HfFileSystem, hf_hub_download
from concurrent.futures import ThreadPoolExecutor

REPO, N_SHARDS = "huggan/wikiart", 72
SHARD_FMT = "data/train-{:05d}-of-00072.parquet"

def scan_shards():
    # Read ONLY the artist column from every shard to locate our painters.
    def one(i):
        fs = HfFileSystem()
        for attempt in range(3):
            try:
                with fs.open(f"datasets/{REPO}/{SHARD_FMT.format(i)}", "rb") as f:
                    col = pq.ParquetFile(f).read(columns=["artist"]).column("artist").to_pylist()
                return i, collections.Counter(col)
            except Exception:
                if attempt == 2:
                    raise
                time.sleep(2)
    with ThreadPoolExecutor(16) as ex:
        return dict(ex.map(one, range(N_SHARDS)))

if CACHE_IMG.exists():
    print("cache already built - skipping scan")
    needed = []
else:
    counts = scan_shards()
    needed = sorted(i for i, c in counts.items() if any(c.get(a, 0) for a in ARTISTS))
    total  = sum(c.get(a, 0) for c in counts.values() for a in ARTISTS)
    per    = collections.Counter()
    for c in counts.values():
        for a, n in ARTISTS.items():
            per[n] += c.get(a, 0)
    print(f"{total} paintings across {len(needed)} shards (~{len(needed)*0.47:.1f} GB to download)")
    print(dict(per))

In [ ]:
def genre_name(raw_names, idx):
    if idx is None or idx < 0 or idx >= len(raw_names):
        return "other"
    g = raw_names[idx].lower()
    return g if g in GENRE_ID else "other"

def fetch_shard(shard, retries=6):
    # Half-gigabyte downloads get reset by the server often enough that a bare
    # hf_hub_download will eventually kill a 16-shard run. Retry with backoff.
    for k in range(retries):
        try:
            return hf_hub_download(REPO, SHARD_FMT.format(shard),
                                   repo_type="dataset", local_dir=TMP)
        except Exception as e:
            if k == retries - 1:
                raise
            wait = 2 ** k
            print(f"  shard {shard}: {type(e).__name__}, retrying in {wait}s "
                  f"({k+1}/{retries-1})", flush=True)
            time.sleep(wait)

def extract_shard(shard, raw_genres):
    # Download one shard, write out matching paintings as JPEGs, delete the shard.
    TMP.mkdir(exist_ok=True)
    path = fetch_shard(shard)
    rows = []
    try:
        tbl = pq.read_table(path, columns=["image", "artist", "genre"])
        imgs, arts, gens = tbl.column("image"), tbl.column("artist"), tbl.column("genre")
        for i in range(tbl.num_rows):
            aid = arts[i].as_py()
            if aid not in ARTISTS:
                continue
            artist = ARTISTS[aid]
            g = genre_name(raw_genres, gens[i].as_py())
            rec = imgs[i].as_py()
            data = rec["bytes"] if isinstance(rec, dict) else rec
            try:
                im = Image.open(io.BytesIO(data)).convert("RGB")
            except Exception:
                continue
            w, h = im.size
            s = ARCHIVE_SIDE / min(w, h)
            if s < 1.0:                       # only ever downscale
                im = im.resize((max(1, round(w*s)), max(1, round(h*s))), Image.LANCZOS)
            d = ARCHIVE / artist; d.mkdir(exist_ok=True)
            fn = d / f"{shard:05d}_{i:05d}__{g}.jpg"
            im.save(fn, quality=92)
            rows.append({"file": str(fn.relative_to(ROOT)), "artist": artist, "genre": g})
    finally:
        Path(path).unlink(missing_ok=True)    # free the 470 MB immediately
    return rows

if needed:
    # WikiArt's genre ClassLabel names, in label order.
    RAW_GENRES = ["abstract_painting", "cityscape", "genre_painting", "illustration",
                  "landscape", "nude_painting", "portrait", "religious_painting",
                  "sketch_and_study", "still_life", "unknown genre"]

    man = json.loads(MANIFEST.read_text()) if MANIFEST.exists() else {"done": [], "rows": []}
    todo = [s for s in needed if s not in man["done"]]
    print(f"{len(man['done'])} shards already done, {len(todo)} to go")

    failed = []
    for s in tqdm(todo, desc="shards"):
        try:
            man["rows"] += extract_shard(s, RAW_GENRES)
            man["done"].append(s)
            MANIFEST.write_text(json.dumps(man))      # checkpoint after every shard
        except Exception as e:
            failed.append(s)
            print(f"  shard {s} gave up: {type(e).__name__}", flush=True)

    if failed:
        print(f"{len(failed)} shard(s) failed: {failed}")
        print("Re-run this cell to retry just those - finished shards are skipped.")
    else:
        shutil.rmtree(TMP, ignore_errors=True)        # only clean up on a full pass
        print("extracted", len(man["rows"]), "paintings")

### 3b. Pack into a training cache

The JPEG archive in `data/paintings/` is the durable copy — browse it in Finder,
and rebuild a higher-resolution cache from it later without re-downloading.

Here we resize each painting to short-side 80, centre-crop to 80×80, and stack
everything into a single ~100 MB `uint8` array. Small enough to sit in RAM for
the whole run, which removes the dataloader from the critical path entirely.

In [ ]:
def build_cache():
    man = json.loads(MANIFEST.read_text())
    rows = man["rows"]

    # Fold any genre that is too rare to learn into "other".
    gc = collections.Counter(r["genre"] for r in rows)
    keep = {g for g, n in gc.items() if n >= MIN_GENRE_COUNT and g in GENRE_ID}
    print("genre counts:", dict(gc.most_common()))
    print("kept as their own class:", sorted(keep))

    imgs = np.zeros((len(rows), CACHE_SIZE, CACHE_SIZE, 3), dtype=np.uint8)
    labs = np.zeros((len(rows), 2), dtype=np.int64)
    ok = 0
    for r in tqdm(rows, desc="packing"):
        try:
            im = Image.open(ROOT / r["file"]).convert("RGB")
        except Exception:
            continue
        w, h = im.size
        s = CACHE_SIZE / min(w, h)
        im = im.resize((max(CACHE_SIZE, round(w*s)), max(CACHE_SIZE, round(h*s))), Image.LANCZOS)
        w, h = im.size
        left, top = (w - CACHE_SIZE) // 2, (h - CACHE_SIZE) // 2
        im = im.crop((left, top, left + CACHE_SIZE, top + CACHE_SIZE))
        imgs[ok] = np.asarray(im, dtype=np.uint8)
        g = r["genre"] if r["genre"] in keep else "other"
        labs[ok] = (ARTIST_ID[r["artist"]], GENRE_ID[g])
        ok += 1
    imgs, labs = imgs[:ok], labs[:ok]
    np.save(CACHE_IMG, imgs); np.save(CACHE_LAB, labs)
    return imgs, labs

if CACHE_IMG.exists():
    images, labels = np.load(CACHE_IMG), np.load(CACHE_LAB)
    print("loaded cache from disk")
else:
    images, labels = build_cache()

print("images", images.shape, images.dtype, f"{images.nbytes/1e6:.0f} MB")
for i, n in enumerate(ARTIST_NAMES):
    sub = labels[labels[:, 0] == i]
    gs = collections.Counter(GENRES[g] for g in sub[:, 1])
    print(f"  {n:10s} {len(sub):5d}  {dict(gs.most_common(4))}")

## 4. Batches and augmentation

Two augmentations, both cheap and both applied on the fly:

- **random 64×64 crop** out of the cached 80×80 (16 px of translation jitter)
- **random horizontal flip**

With only ~5k paintings, augmentation is doing real work here — it is most of
what stands between us and memorisation.

In [ ]:
images_t = torch.from_numpy(images)            # uint8 [N,80,80,3], stays on CPU
labels_t = torch.from_numpy(labels)            # int64 [N,2]
N = len(images_t)

def get_batch(bs=BATCH_SIZE):
    idx = torch.randint(0, N, (bs,))
    raw = images_t[idx]
    m = CACHE_SIZE - IMAGE_SIZE
    ox = torch.randint(0, m + 1, (bs,))
    oy = torch.randint(0, m + 1, (bs,))
    out = torch.empty(bs, IMAGE_SIZE, IMAGE_SIZE, 3, dtype=torch.uint8)
    for i in range(bs):                        # ~0.5 ms for bs=128, not worth vectorising
        out[i] = raw[i, oy[i]:oy[i]+IMAGE_SIZE, ox[i]:ox[i]+IMAGE_SIZE]
    x = out.permute(0, 3, 1, 2).float().div_(127.5).sub_(1.0)   # -> [-1,1], NCHW
    flip = torch.rand(bs) < 0.5
    x[flip] = torch.flip(x[flip], dims=[3])
    return x.to(DEVICE), labels_t[idx].to(DEVICE)

def show(x, nrow=8, title=None, figsize=(10, 5)):
    # x: float tensor in [-1,1], NCHW
    from torchvision.utils import make_grid
    g = make_grid(x.detach().float().cpu().clamp(-1, 1).add(1).div(2), nrow=nrow, padding=2)
    plt.figure(figsize=figsize)
    plt.imshow(g.permute(1, 2, 0).numpy()); plt.axis("off")
    if title: plt.title(title)
    plt.show()

xb, yb = get_batch(16)
show(xb, nrow=8, title="real paintings (augmented 64x64 crops)")
print([f"{ARTIST_NAMES[a]}/{GENRES[g]}" for a, g in yb.tolist()[:8]])

## 5. The forward process

A cosine noise schedule (Nichol & Dhariwal). Compared to the original linear
schedule it destroys information more gradually, which matters a lot at 64px —
a linear schedule spends too many of its steps on images that are already pure
noise.

The forward process has a closed form, so we can jump straight to any timestep:

$$x_t = \sqrt{\bar\alpha_t}\,x_0 + \sqrt{1-\bar\alpha_t}\,\epsilon$$

In [ ]:
def cosine_alpha_bar(T, s=0.008):
    t = torch.linspace(0, T, T + 1, dtype=torch.float64) / T
    f = torch.cos((t + s) / (1 + s) * math.pi / 2) ** 2
    return (f / f[0]).clamp(1e-8, 1.0)

ab_full = cosine_alpha_bar(TIMESTEPS)          # length T+1, ab_full[0] == 1
betas = (1 - ab_full[1:] / ab_full[:-1]).clamp(0, 0.999)
alphas_bar = torch.cumprod(1 - betas, 0).float().to(DEVICE)   # index t = timestep t

sqrt_ab     = alphas_bar.sqrt()
sqrt_1mab   = (1 - alphas_bar).sqrt()

def q_sample(x0, t, noise):
    a = sqrt_ab[t].view(-1, 1, 1, 1)
    b = sqrt_1mab[t].view(-1, 1, 1, 1)
    return a * x0 + b * noise

plt.figure(figsize=(9, 3))
plt.plot(alphas_bar.cpu(), label="alpha_bar (signal)")
plt.plot((1 - alphas_bar).cpu(), label="1 - alpha_bar (noise)")
plt.xlabel("timestep"); plt.legend(); plt.title("cosine schedule"); plt.grid(alpha=.3); plt.show()

# What the forward process actually looks like on one painting.
x0 = get_batch(1)[0].repeat(8, 1, 1, 1)
ts = torch.linspace(0, TIMESTEPS - 1, 8).long().to(DEVICE)
show(q_sample(x0, ts, torch.randn_like(x0)), nrow=8,
     title="forward diffusion: t = " + ", ".join(str(int(t)) for t in ts))

## 6. The UNet

Standard DDPM backbone: residual blocks with GroupNorm + SiLU, self-attention at
the 16² and 8² resolutions, and skip connections from encoder to decoder.

Conditioning is **additive in the embedding space** — the timestep embedding,
artist embedding and genre embedding are summed into one vector, which each
residual block turns into a per-channel scale and shift (FiLM). Both label
embeddings carry one extra "null" slot; dropping a label to null during training
is what makes classifier-free guidance possible at sampling time.

### Sizing this for 16 GB

Activation memory is dominated by the **widest layer at full 64×64 resolution**,
not by the parameter count. That makes the textbook DDPM config a trap here —
measured on this machine:

| base | mults | params | activations @ batch 64 |
|---|---|---|---|
| 128 | (1,2,2,4) | 77.5M | ~6.0 GB → **OOM in practice** |
| 128 | (1,2,2,2) | 39.6M | ~5.9 GB |
| 96  | (1,2,2,2) | 22.3M | ~4.5 GB |
| **64** | **(1,2,3,4)** | **23.0M** | **~3.1 GB** ← what we use |
| 64  | (1,2,2,2) |  9.9M | ~3.0 GB |

(Activations shown at batch 64 to compare architectures on equal terms; we
actually train at batch 16. The ranking is the point.)

`base=64, (1,2,3,4)` gets essentially the same parameter count as `base=96` for
two-thirds of the memory, by staying narrow at 64×64 and widening only in the
deeper levels where feature maps are 8× smaller. The first row is not
hypothetical — it allocated 19.8 GB and died.

Going from `(1,2,2,2)` to `(1,2,3,4)` costs almost no time, because the extra
width lands at the 16² and 8² levels where the feature maps are tiny — over 80%
of the FLOPs are in the two shallowest levels, which are identical in both. It
is 2.3× the parameters for roughly free.

In [ ]:
def timestep_embedding(t, dim):
    half = dim // 2
    freqs = torch.exp(-math.log(10000) * torch.arange(half, device=t.device).float() / half)
    a = t.float()[:, None] * freqs[None]
    return torch.cat([torch.cos(a), torch.sin(a)], dim=-1)

def gn(ch):
    g = 32                      # largest group count <= 32 that divides ch
    while ch % g:
        g //= 2
    return nn.GroupNorm(g, ch)

class ResBlock(nn.Module):
    def __init__(self, cin, cout, emb, dropout):
        super().__init__()
        self.norm1 = gn(cin)
        self.conv1 = nn.Conv2d(cin, cout, 3, padding=1)
        self.emb   = nn.Linear(emb, cout * 2)          # FiLM scale + shift
        self.norm2 = gn(cout)
        self.drop  = nn.Dropout(dropout)
        self.conv2 = nn.Conv2d(cout, cout, 3, padding=1)
        self.skip  = nn.Conv2d(cin, cout, 1) if cin != cout else nn.Identity()
        nn.init.zeros_(self.conv2.weight); nn.init.zeros_(self.conv2.bias)

    def forward(self, x, emb):
        h = self.conv1(F.silu(self.norm1(x)))
        scale, shift = self.emb(F.silu(emb))[:, :, None, None].chunk(2, dim=1)
        h = self.norm2(h) * (1 + scale) + shift
        h = self.conv2(self.drop(F.silu(h)))
        return h + self.skip(x)

class AttnBlock(nn.Module):
    def __init__(self, ch, heads):
        super().__init__()
        self.norm, self.heads = gn(ch), heads
        self.qkv  = nn.Conv2d(ch, ch * 3, 1)
        self.proj = nn.Conv2d(ch, ch, 1)
        nn.init.zeros_(self.proj.weight); nn.init.zeros_(self.proj.bias)

    def forward(self, x):
        B, C, H, W = x.shape
        q, k, v = self.qkv(self.norm(x)).reshape(
            B, 3, self.heads, C // self.heads, H * W).unbind(1)
        q, k, v = (t.transpose(-1, -2) for t in (q, k, v))     # B,heads,HW,dim
        h = F.scaled_dot_product_attention(q, k, v)
        h = h.transpose(-1, -2).reshape(B, C, H, W)
        return x + self.proj(h)

class Down(nn.Module):
    def __init__(self, ch):
        super().__init__(); self.op = nn.Conv2d(ch, ch, 3, stride=2, padding=1)
    def forward(self, x): return self.op(x)

class Up(nn.Module):
    def __init__(self, ch):
        super().__init__(); self.op = nn.Conv2d(ch, ch, 3, padding=1)
    def forward(self, x):
        return self.op(F.interpolate(x, scale_factor=2, mode="nearest"))

class Stage(nn.Module):
    # one residual block plus optional attention, so the skip bookkeeping stays flat
    def __init__(self, res, attn=None):
        super().__init__(); self.res, self.attn = res, attn
    def forward(self, x, emb):
        x = self.res(x, emb)
        return self.attn(x) if self.attn is not None else x

In [ ]:
class UNet(nn.Module):
    def __init__(self):
        super().__init__()
        emb = BASE_CH * 4
        self.emb_dim = emb
        self.time_mlp = nn.Sequential(nn.Linear(BASE_CH, emb), nn.SiLU(), nn.Linear(emb, emb))
        self.artist_emb = nn.Embedding(N_ARTISTS + 1, emb)   # +1 = null token for CFG
        self.genre_emb  = nn.Embedding(N_GENRES + 1, emb)
        nn.init.normal_(self.artist_emb.weight, std=0.02)
        nn.init.normal_(self.genre_emb.weight,  std=0.02)

        self.stem = nn.Conv2d(3, BASE_CH, 3, padding=1)
        self.down, chans, ch, res = nn.ModuleList(), [BASE_CH], BASE_CH, IMAGE_SIZE
        for i, m in enumerate(CHANNEL_MULT):
            out = BASE_CH * m
            for _ in range(NUM_RES):
                self.down.append(Stage(ResBlock(ch, out, emb, DROPOUT),
                                       AttnBlock(out, NUM_HEADS) if res in ATTN_RES else None))
                ch = out; chans.append(ch)
            if i != len(CHANNEL_MULT) - 1:
                self.down.append(Down(ch)); chans.append(ch); res //= 2

        self.mid1 = ResBlock(ch, ch, emb, DROPOUT)
        self.mid_attn = AttnBlock(ch, NUM_HEADS)
        self.mid2 = ResBlock(ch, ch, emb, DROPOUT)

        self.up = nn.ModuleList()
        for i, m in reversed(list(enumerate(CHANNEL_MULT))):
            out = BASE_CH * m
            for j in range(NUM_RES + 1):
                self.up.append(Stage(ResBlock(ch + chans.pop(), out, emb, DROPOUT),
                                     AttnBlock(out, NUM_HEADS) if res in ATTN_RES else None))
                ch = out
                if i and j == NUM_RES:
                    self.up.append(Up(ch)); res *= 2

        self.out = nn.Sequential(gn(ch), nn.SiLU(), nn.Conv2d(ch, 3, 3, padding=1))
        nn.init.zeros_(self.out[-1].weight); nn.init.zeros_(self.out[-1].bias)

    def forward(self, x, t, artist, genre):
        emb = (self.time_mlp(timestep_embedding(t, BASE_CH))
               + self.artist_emb(artist) + self.genre_emb(genre))
        h = self.stem(x); hs = [h]
        for mod in self.down:
            h = mod(h, emb) if isinstance(mod, Stage) else mod(h)
            hs.append(h)
        h = self.mid2(self.mid_attn(self.mid1(h, emb)), emb)
        for mod in self.up:
            if isinstance(mod, Stage):
                h = mod(torch.cat([h, hs.pop()], dim=1), emb)
            else:
                h = mod(h)
        return self.out(h)

model = UNet().to(DEVICE)
nparam = sum(p.numel() for p in model.parameters())
print(f"UNet: {nparam/1e6:.1f}M parameters")

with torch.no_grad():
    o = model(torch.randn(2, 3, IMAGE_SIZE, IMAGE_SIZE, device=DEVICE),
              torch.tensor([0, 999], device=DEVICE),
              torch.tensor([0, NULL_ARTIST], device=DEVICE),
              torch.tensor([0, NULL_GENRE], device=DEVICE))
print("forward ok:", tuple(o.shape))

## 7. The loss

We predict **v** rather than the noise. With
$a=\sqrt{\bar\alpha_t}$, $b=\sqrt{1-\bar\alpha_t}$:

$$v = a\,\epsilon - b\,x_0 \qquad\Longrightarrow\qquad x_0 = a\,x_t - b\,v,\quad \epsilon = b\,x_t + a\,v$$

v-prediction keeps the target well-scaled at *every* timestep. Plain
ε-prediction gives a nearly trivial target as $t\to 0$ and interacts badly with
strong guidance, which is exactly the regime we want to sample in. Set
`PRED_TYPE = "eps"` in section 2 to compare.

**Label dropout** replaces artist and/or genre with the null token so the same
network learns the conditional *and* unconditional scores. The mixture below
deliberately trains all three combinations that guidance needs at sampling time:
`(null,null)`, `(artist,null)` and `(artist,genre)`.

In [ ]:
def drop_labels(y):
    # y: [B,2] int64 (artist, genre). Returns a CFG-ready copy.
    a, g = y[:, 0].clone(), y[:, 1].clone()
    r = torch.rand(len(y), device=y.device)
    drop_both   = r < 0.10                    # 10% -> (null, null),   fully unconditional
    drop_genre  = (r >= 0.10) & (r < 0.20)    # 10% -> (artist, null), artist-only
    drop_artist = (r >= 0.20) & (r < 0.25)    #  5% -> (null, genre),  genre-only
    a[drop_both | drop_artist] = NULL_ARTIST  # the other 75% keep both labels
    g[drop_both | drop_genre]  = NULL_GENRE
    return a, g

def diffusion_loss(model, x0, y):
    B = x0.shape[0]
    t = torch.randint(0, TIMESTEPS, (B,), device=x0.device)
    noise = torch.randn_like(x0)
    xt = q_sample(x0, t, noise)
    a = sqrt_ab[t].view(-1, 1, 1, 1)
    b = sqrt_1mab[t].view(-1, 1, 1, 1)
    target = (a * noise - b * x0) if PRED_TYPE == "v" else noise
    art, gen = drop_labels(y)
    pred = model(xt, t, art, gen)
    return F.mse_loss(pred, target)

xb, yb = get_batch(8)
print("loss at init:", diffusion_loss(model, xb, yb).item())

### Measured on this machine (M2 Pro, 16 GB, 23M params, 64px)

| batch | ms/step | img/s | MPS allocated |
|---|---|---|---|
| 8  | 295 | 27 | 2.2 GB |
| **16** | **500** | **32** | **3.4 GB** |
| 24 | 684 | 35 | 5.4 GB |
| 32 | 911 | 35 | 6.5 GB |
| 64 | **54,000** | 1 | thrashing — 20 GB wanted, ~0.2 GB free |

Throughput is basically flat from 16 upward, so the larger batches buy nothing
except OOM risk. `bf16` autocast was also tested and is **slower** on MPS
(835 ms vs 600 ms at batch 16) with no memory saving, so we stay in fp32.

If you close your browser you can push `BATCH_SIZE` to 32 for ~10% more
throughput. Not worth it if it risks dying four hours in.

In [ ]:
# Quick throughput benchmark so you know the ETA before committing hours.
opt_tmp = torch.optim.AdamW(model.parameters(), lr=1e-6)
xb, yb = get_batch(BATCH_SIZE)
for _ in range(3):                                  # warm up MPS kernels
    diffusion_loss(model, xb, yb).backward(); opt_tmp.zero_grad()
sync()

t0, ITERS = time.time(), 10
for _ in range(ITERS):
    loss = diffusion_loss(model, *get_batch(BATCH_SIZE))
    loss.backward(); opt_tmp.step(); opt_tmp.zero_grad()
sync()
dt = (time.time() - t0) / ITERS

print(f"{dt*1000:.0f} ms/step at batch {BATCH_SIZE}  ->  {1/dt:.2f} steps/s")
print(f"{TOTAL_STEPS} steps ~= {TOTAL_STEPS*dt/3600:.1f} hours")
del opt_tmp

## 8. Sampling

**DDIM** with 50 steps instead of the full 1,000 — deterministic (`eta=0`) and
about 20× faster, with no meaningful quality loss at this scale.

**Two-axis classifier-free guidance.** The usual CFG formula has one conditional
and one unconditional prediction. To steer artist and genre *independently* we
use three predictions and compose them:

$$\hat v = v_{\varnothing} + w_a\,(v_{a} - v_{\varnothing}) + w_g\,(v_{a,g} - v_{a})$$

The first term pulls toward the artist, the second adds the genre *on top of*
that artist. All three run as one batched forward pass. Setting `w_a == w_g`
recovers ordinary joint guidance.

Guidance strength is a real trade-off: low (1–2) is diverse but muddy, high
(5+) is vivid and over-saturated. 3 is a good starting point.

In [ ]:
@torch.no_grad()
def sample(n=16, artist=None, genre=None, w_artist=3.0, w_genre=2.0,
           steps=50, eta=0.0, net=None, seed=None, artist_mix=None):
    # artist / genre: name string, or None for unconditional on that axis.
    # artist_mix: dict like {"monet": 0.5, "van_gogh": 0.5} to blend two painters.
    net = net or ema_model
    net.eval()
    if seed is not None:
        torch.manual_seed(seed)

    a_id = NULL_ARTIST if artist is None else ARTIST_ID[artist]
    g_id = NULL_GENRE  if genre  is None else GENRE_ID[genre]
    A_null = torch.full((n,), NULL_ARTIST, device=DEVICE, dtype=torch.long)
    G_null = torch.full((n,), NULL_GENRE,  device=DEVICE, dtype=torch.long)
    A_cond = torch.full((n,), a_id, device=DEVICE, dtype=torch.long)
    G_cond = torch.full((n,), g_id, device=DEVICE, dtype=torch.long)

    ts = torch.linspace(TIMESTEPS - 1, 0, steps).long().to(DEVICE)
    x = torch.randn(n, 3, IMAGE_SIZE, IMAGE_SIZE, device=DEVICE)

    for i, t in enumerate(tqdm(ts, desc="ddim", leave=False)):
        tb = t.repeat(n)
        # one batched forward over the three conditionings
        x3 = torch.cat([x, x, x], 0)
        t3 = torch.cat([tb, tb, tb], 0)
        a3 = torch.cat([A_null, A_cond, A_cond], 0)
        g3 = torch.cat([G_null, G_null, G_cond], 0)

        if artist_mix:
            out = 0
            for name, wt in artist_mix.items():
                a3m = a3.clone()
                a3m[n:] = ARTIST_ID[name]
                out = out + wt * net(x3, t3, a3m, g3)
            pred = out / sum(artist_mix.values())
        else:
            pred = net(x3, t3, a3, g3)

        p_none, p_art, p_both = pred.chunk(3, 0)
        p = p_none + w_artist * (p_art - p_none) + w_genre * (p_both - p_art)

        ab_t = alphas_bar[t]
        ab_prev = alphas_bar[ts[i + 1]] if i + 1 < len(ts) else torch.tensor(1.0, device=DEVICE)
        a_t, b_t = ab_t.sqrt(), (1 - ab_t).sqrt()

        if PRED_TYPE == "v":
            x0 = a_t * x - b_t * p
            eps = b_t * x + a_t * p
        else:
            eps = p
            x0 = (x - b_t * eps) / a_t
        x0 = x0.clamp(-1, 1)

        sigma = eta * ((1 - ab_prev) / (1 - ab_t)).sqrt() * (1 - ab_t / ab_prev).sqrt()
        dir_xt = (1 - ab_prev - sigma ** 2).clamp(min=0).sqrt() * eps
        x = ab_prev.sqrt() * x0 + dir_xt
        if eta > 0 and i + 1 < len(ts):
            x = x + sigma * torch.randn_like(x)
    return x

# A dedicated model instance that always holds the EMA weights.
ema_model = UNet().to(DEVICE)

def load_ema(into=None):
    into = into or ema_model
    ck = torch.load(CKPT_DIR / "latest.pt", map_location=DEVICE)
    into.load_state_dict({k: v.to(DEVICE) for k, v in ck["ema"].items()})
    into.eval()
    return ck["step"]

def preview(ema_obj):
    # 4 rows of 4, one row per artist; used for the inline previews during training
    ema_model.load_state_dict({k: v.to(DEVICE) for k, v in ema_obj.shadow.items()})
    ema_model.eval()
    outs = [sample(4, artist=a, genre="landscape", steps=30, net=ema_model)
            for a in ARTIST_NAMES]
    return torch.cat(outs, 0)

## 9. Training

Resumable: the cell saves `checkpoints/latest.pt` every 2,000 steps and reloads
it on re-run, so interrupting the kernel costs at most 2,000 steps.

Two things that matter more than they look:

- **EMA.** We keep an exponential moving average of the weights (decay 0.9995)
  and sample from *that*, never from the raw weights. Diffusion samples from raw
  weights are visibly worse; this is not an optional refinement. Note the
  **warmup** inside `EMA.update` — without it the average stays anchored to the
  random initialisation for thousands of steps and every early preview is pure
  noise even while the raw model is already painting. That is a real trap: it
  looks exactly like a broken sampler.
- **Zero-init output layers.** Each residual block and the final conv start at
  zero, so the network begins as an identity map and early training is stable.

Inline previews render every 1,000 steps.

**What to realistically expect.** 24k steps at batch 16 is ~384k images seen —
roughly 74 passes over the dataset, but around 250× less total compute than the
original DDPM used on CIFAR-10. So: colour fields and rough palette separation by
3k, brushstroke texture and horizon/sky structure by 8k, and blobby-but-plausible
impressionist compositions by 20k. You will **not** get sharp faces or objects at
this budget — the win here is that the artist and genre conditioning visibly
works, and that every line of it is yours.

To push further, raise `TOTAL_STEPS` and re-run. Note the cosine LR schedule is
defined against `TOTAL_STEPS`, so decide before a long run rather than during it.

In [ ]:
class EMA:
    def __init__(self, model, decay):
        self.decay, self.n = decay, 0
        self.shadow = {k: v.detach().clone().float() for k, v in model.state_dict().items()}

    @torch.no_grad()
    def update(self, model):
        self.n += 1
        # Warmup matters more than it looks. The shadow starts at the RANDOM
        # INIT, so a flat 0.9995 leaves it 0.9995^600 = 74% initialisation even
        # after 600 steps. The zero-init output layers get scaled down to a
        # quarter of their trained value and samples come out as pure RGB noise
        # while the raw weights already look fine. This grows the averaging
        # horizon from ~10 steps up to 1/(1-decay) instead of starting there.
        d = min(self.decay, (1 + self.n) / (10 + self.n))
        for k, v in model.state_dict().items():
            if v.dtype.is_floating_point:
                self.shadow[k].mul_(d).add_(v.detach().float(), alpha=1 - d)
            else:
                self.shadow[k].copy_(v)

    def copy_to(self, model):
        model.load_state_dict(self.shadow, strict=True)

def lr_at(step):
    if step < WARMUP:
        return LR * step / WARMUP
    p = (step - WARMUP) / max(1, TOTAL_STEPS - WARMUP)
    return LR * 0.5 * (1 + math.cos(math.pi * min(1.0, p)))

model = UNet().to(DEVICE)
opt   = torch.optim.AdamW(model.parameters(), lr=LR, betas=(0.9, 0.99), weight_decay=0.0)
ema   = EMA(model, EMA_DECAY)
step, history = 0, []

LATEST = CKPT_DIR / "latest.pt"
if LATEST.exists():
    ck = torch.load(LATEST, map_location=DEVICE)
    model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"])
    ema.shadow = {k: v.to(DEVICE) for k, v in ck["ema"].items()}
    ema.n = ck.get("ema_n", ck["step"])          # keep the EMA warmup on schedule
    step, history = ck["step"], ck.get("history", [])
    print(f"resumed from step {step}")
else:
    print("starting fresh")

In [ ]:
# Re-run this cell to continue training from wherever it stopped.
pbar = tqdm(initial=step, total=TOTAL_STEPS, desc="train")
model.train()
running = []

try:
    while step < TOTAL_STEPS:
        for g in opt.param_groups:
            g["lr"] = lr_at(step)

        x0, y = get_batch(BATCH_SIZE)
        loss = diffusion_loss(model, x0, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        opt.step(); opt.zero_grad(set_to_none=True)
        ema.update(model)
        step += 1
        running.append(loss.item())

        if step % LOG_EVERY == 0:
            m = float(np.mean(running)); running = []
            history.append((step, m))
            pbar.set_postfix(loss=f"{m:.4f}", lr=f"{lr_at(step):.1e}")
        pbar.update(1)

        if step % CKPT_EVERY == 0 or step == TOTAL_STEPS:
            torch.save({"model": model.state_dict(), "opt": opt.state_dict(),
                        "ema": ema.shadow, "ema_n": ema.n,
                        "step": step, "history": history}, LATEST)

        if step % SAMPLE_EVERY == 0 or step == TOTAL_STEPS:
            model.eval()
            grid = preview(ema)
            show(grid, nrow=4, title=f"EMA samples @ step {step}", figsize=(9, 9))
            model.train()
finally:
    pbar.close()
    torch.save({"model": model.state_dict(), "opt": opt.state_dict(),
                "ema": ema.shadow, "ema_n": ema.n,
                "step": step, "history": history}, LATEST)
    print("checkpoint saved at step", step)

## 10. Explore the trained model

In [ ]:
step_loaded = load_ema()
print("sampling from EMA weights at step", step_loaded)

for a in ARTIST_NAMES:
    x = sample(8, artist=a, genre="landscape", w_artist=3.0, w_genre=2.0, seed=0)
    show(x, nrow=8, title=f"{a} / landscape", figsize=(14, 2.2))

### Guidance sweep — find the strength you like

In [ ]:
for w in [1.0, 2.0, 3.0, 5.0, 7.0]:
    x = sample(8, artist="van_gogh", genre="landscape", w_artist=w, w_genre=2.0, seed=1)
    show(x, nrow=8, title=f"van_gogh / landscape, w_artist={w}", figsize=(14, 2.2))

### Genre control — same painter, different subject

In [ ]:
for g in ["landscape", "portrait", "still_life", "cityscape"]:
    x = sample(8, artist="van_gogh", genre=g, w_artist=3.0, w_genre=3.0, seed=2)
    show(x, nrow=8, title=f"van_gogh / {g}", figsize=(14, 2.2))

### Blending two painters

In [ ]:
x = sample(8, artist="monet", genre="landscape", seed=3,
           artist_mix={"monet": 0.5, "van_gogh": 0.5}, w_artist=3.0, w_genre=2.0)
show(x, nrow=8, title="0.5 monet + 0.5 van_gogh / landscape", figsize=(14, 2.2))

### Loss curve

In [ ]:
ck = torch.load(CKPT_DIR / "latest.pt", map_location="cpu")
h = np.array(ck["history"])
plt.figure(figsize=(9, 3))
plt.plot(h[:, 0], h[:, 1], lw=.8)
plt.xlabel("step"); plt.ylabel("loss"); plt.grid(alpha=.3)
plt.title(f"training loss ({ck['step']} steps)"); plt.show()

### Save a batch of favourites to `samples/`

In [ ]:
from torchvision.utils import save_image
out = OUT_DIR / f"step{step_loaded}"
out.mkdir(parents=True, exist_ok=True)
for a in ARTIST_NAMES:
    for g in ["landscape", "portrait"]:
        x = sample(16, artist=a, genre=g, w_artist=3.0, w_genre=2.0)
        save_image(x.add(1).div(2).clamp(0, 1), out / f"{a}__{g}.png", nrow=8, padding=2)
print("wrote", out)

## Where to go next

- **Bump the resolution.** Set `IMAGE_SIZE = 128`, `CACHE_SIZE = 144`, drop
  `BATCH_SIZE` to **4** (128px is 4× the activations per image — batch 16 will
  thrash), delete `data/cache_images.npy` and re-run section 3b. The
  JPEG archive is already stored at short side 144, so there is nothing to
  re-download. Warm-starting from the 64px checkpoint converges much faster than
  starting over — the conv stack is resolution-agnostic.
- **More painters.** Add ids to `ARTISTS` in section 2 and delete
  `data/manifest.json`. Check the shard cost first: artists spread across many
  shards get expensive fast.
- **Better sampling.** Try `eta=0.2` for a little stochasticity, or raise
  `steps` to 100–250.
- **Watch for memorisation.** With ~5k images this model can start copying. Take
  a sample you like, and nearest-neighbour it against the training cache to
  confirm the model invented it.